# Annotation listing and navigation

This notebook imports pUC19 from a GenBank file, loads its feature annotations,
and demonstrates two ways to work with them:

1. **Without a viewer** — query annotations directly from a `SequenceGraph` (preferred).
2. **With a widget** — navigate to a feature and add highlights interactively.

In [ ]:
import os
import tempfile
import gen

## Import pUC19

Create a fresh in-memory repository and import a GenBank file.

In [ ]:
# Adjust this path if running from outside the project root.
FIXTURE = os.path.abspath("../../fixtures/puc19.gb")

tmp = tempfile.mkdtemp()
repo = gen.Repository(os.path.join(tmp, ".gen"))
sample = repo.import_genbank(FIXTURE)

sg = sample[0]
print("Sequence graph:", sg.name)


## List annotations without opening a viewer

`SequenceGraph.list_annotations()` reads persisted annotations directly from the
database — no widget required.  This is the preferred way to programmatically
inspect, filter, or iterate over features stored in the graph.

In [ ]:
anns = sg.list_annotations()
print(f"{len(anns)} annotations stored in graph")
for a in anns:
    print(a)


## Visualise with a widget

Call `.plot()` to open an interactive widget.  Annotations from the GenBank
file are loaded automatically as inline graph highlights with floating labels.

Additional GFF3 or BED files can be added at any time:

```python
fig.add_annotation_track(file="path/to/features.gff3")
```

To start with a blank canvas:

```python
fig.clear_all_annotations()
```

In [ ]:
fig = sg.plot(rows=24)
fig


## List widget annotations

`fig.list_annotations()` returns the annotations currently tracked by the widget —
this includes both the stored database annotations and any highlights added
programmatically with `fig.add_annotation()`.

In [ ]:
anns = fig.list_annotations()
print(f"{len(anns)} annotations loaded")
for a in anns:
    print(a)

## Navigate to the MCS

Filter for the MCS feature and navigate to it two ways:

* `widget.go_to(ann)` — left-pins the annotation start at column 12, no highlight
* `widget.show(ann)` — left-pins the annotation start and adds a highlight

In [ ]:
# Left-pin the MCS start at column 12 from the left edge.
#mcs = next(a for a in anns if a.name == "MCS")
#fig.go_to(mcs)
#fig


In [ ]:
# Or go somewhere and immediately add a highlight.
#pbla = next(a for a in anns if a.name == "AmpR promoter")

#fig.show(pbla)
#fig

## Search, filter, and navigate

Search for a sequence, wrap matches as `Annotation` objects, then navigate to one.

In [ ]:
fig = sg.plot(rows=24)

# Search for Dcm methylation sites and add matches to the widget.
results = sg.search("CCWGG", "dna")

# The search query is a palindrome, so every match will also be a match on the opposite strand,
# filter by strand to prevent double labels:
matches = [locus for locus in results if locus.strand == "+"]
print(f"{len(matches)} match(es) found")

for i, locus in enumerate(matches):
    fig.add_annotation(gen.Annotation(locus, f"Dcm ({i + 1} of {len(matches)})"))

fig.go_to(matches[0])
fig

In [ ]:
# The Dcm highlights added above are widget overlays, not stored in the graph.
# Navigate directly using the search results from the cell above.
print(f"{len(matches)} Dcm match(es) found")
fig.go_to(matches[0])
fig